# Overnight experiment: LFA / SE / CBAM / no attention, repeated grouped cross-validation

**What it does**: On the 560 original images (550 images and 4 classes after dropping Pink disease), run 4-fold cross-validation grouped by capture session, repeated 5 times = 20 folds.
In each fold, B0, B0+LFA, B0+SE and B0+CBAM are each trained with 2 fixed seeds, and compared pairwise.

**Order**: First finish "no attention" and "LFA" on all folds (80 runs, about 4-5 hours), then run SE and CBAM (another 80 runs).
It is fine if the session drops in the middle of the night: every finished run is written to Drive, and re-running all cells skips the ones already done.

**Differences from the old code** (all fixes listed in the plan document)
- Random seeds now actually take effect (torch / numpy / random / sampler / dataloader worker)
- The checkpoint is chosen by **macro F1** on the inner validation set, no longer by accuracy
- Sessions are merged by the cross-class rule (60 groups), and Pink disease is dropped
- Original images are resized to 512 px here on the fly (Lanczos, JPEG q92), consistent with the paper's shrink.py

**In the morning**: Run the last cell, "Analysis", and send the output to Claude.

Before running: in the menu, "Runtime > Change runtime type", select **GPU**.

## 1. Mount Drive, check GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'],
                     capture_output=True, text=True).stdout or '⚠ No GPU! Please change the runtime type')

## 2. Settings

In [ ]:
from pathlib import Path

DRIVE_DIR    = Path('/content/drive/MyDrive/p1_reconstructure')   # folder where you uploaded data_original.zip
DATA_ARCHIVE = DRIVE_DIR / 'data_original.zip'
OUT_DIR      = DRIVE_DIR / 'runs_phase0'          # results are written here (saved after every finished run)
LOCAL_DIR    = Path('/content/work')               # Colab local temp directory

DROP_PINK    = True          # drop Pink disease (only 3 sessions)
CROSS_CLASS  = True          # merge sessions by the cross-class rule (a burst split across two classes becomes one group)
N_FOLDS      = 4
N_REPEATS    = 5             # 4 folds x 5 repeats = 20 folds
SEEDS        = [0, 1]        # training seeds per config per fold
PHASES       = [['none', 'lfa'], ['se', 'cbam']]   # finish the first group first, then the second
EPOCHS_S1, EPOCHS_S2 = 15, 15
BATCH        = 16
INNER_VAL_FRAC = 0.1         # hold out 10% of the training fold, by group, as a validation set (for choosing the checkpoint)
PRETRAINED   = True          # can be set to False for local testing
QUICK_TEST   = False         # True = 1 epoch per stage and only 2 folds, for a quick trial run of a few minutes

## 3. Unpack the original images, resize to 512 px, cache on Drive (no need to resize again after a reconnect)

In [ ]:
import io, os, re, shutil, tarfile, zipfile, pickle, time
from PIL import Image

CLASSES_ALL = ['Algal', 'Leaf_rot', 'Phomopsis', 'Pink_disease', 'Root_disease']
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff', '.heic'}
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE = OUT_DIR / 'cache_512.pkl'


def norm_class(part):
    if part.startswith('Leaf_rot'):
        return 'Leaf_rot'
    for c in CLASSES_ALL:
        if part.lower() == c.lower():
            return c
    return None


def shrink_like_paper(raw_bytes):
    """Identical to shrink.py: long side 512, Lanczos, JPEG q92."""
    with Image.open(io.BytesIO(raw_bytes)) as im:
        im.load()
        if im.mode in ('RGBA', 'P', 'LA'):
            im = im.convert('RGB')
        w, h = im.size
        s = 512 / max(w, h)
        if s < 1:
            im = im.resize((max(1, round(w * s)), max(1, round(h * s))), Image.LANCZOS)
        buf = io.BytesIO()
        im.save(buf, 'JPEG', quality=92, optimize=True)
        return buf.getvalue()


if CACHE.exists():
    with open(CACHE, 'rb') as f:
        cache = pickle.load(f)
    print(f'Loaded cache: {len(cache["images"])} images at 512 px')
else:
    LOCAL_DIR.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    print('Copying the archive from Drive to local disk...')
    local_arc = LOCAL_DIR / 'data.arc'
    shutil.copy(DATA_ARCHIVE, local_arc)
    out = LOCAL_DIR / 'data_original'
    if zipfile.is_zipfile(local_arc):
        zipfile.ZipFile(local_arc).extractall(out)
    elif tarfile.is_tarfile(local_arc):
        tarfile.open(local_arc).extractall(out)
    else:
        raise SystemExit('The archive is neither zip nor tar')
    sess_files = sorted(out.rglob('sessions.csv'), key=lambda p: len(p.parts))
    assert sess_files, 'sessions.csv not found in the archive'
    images = {}
    files = [p for p in out.rglob('*') if p.is_file() and p.suffix.lower() in IMG_EXT]
    for i, p in enumerate(files, 1):
        cls = next((norm_class(x) for x in p.relative_to(out).parts[:-1] if norm_class(x)), None)
        if cls is None:
            continue
        images[(cls, p.name)] = shrink_like_paper(p.read_bytes())
        if i % 50 == 0:
            print(f'  Resizing {i}/{len(files)}', end='\r')
    cache = {'images': images, 'sessions_csv': sess_files[0].read_text(encoding='utf-8-sig')}
    with open(CACHE, 'wb') as f:
        pickle.dump(cache, f)
    print(f'\nResizing done: {len(images)} images, took {time.time() - t0:.0f} s, cached to Drive')

assert len(cache['images']) == 560, f'Expected 560 images, got {len(cache["images"])}'

## 4. Session grouping and the 20-fold split (generated once and saved as folds.csv, then reused)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

RE_SEQ = re.compile(r'^(?P<prefix>[A-Za-z_]*?)(?P<num>\d{3,6})\b')


def build_groups(d, gap=3, cross_class=True):
    """Same as build() in grouping_sensitivity.py: bursts are merged when the number gap is <= gap."""
    d = d.copy()
    d['kind'] = d['session'].astype(str).str.split(':').str[0]
    parsed = d['file'].apply(lambda f: RE_SEQ.match(str(f)))
    d['prefix'] = [(m.group('prefix') or 'IMG') if m else None for m in parsed]
    d['num'] = [int(m.group('num')) if m else np.nan for m in parsed]
    seq = d['kind'].isin(['burst', 'single']) & d['num'].notna()
    lab = {}
    key = ['prefix'] if cross_class else ['cls', 'prefix']
    for k, grp in d[seq].groupby(key):
        run, prev = 0, None
        for i, r in grp.sort_values('num').iterrows():
            if prev is not None and r['num'] - prev > gap:
                run += 1
            lab[i] = f'seq:{k}:{run}'
            prev = r['num']
    for i, r in d[~seq].iterrows():
        lab[i] = r['session']
    return pd.Series(lab).reindex(d.index)


meta = pd.read_csv(io.StringIO(cache['sessions_csv']))
meta['cls'] = meta['cls'].map(norm_class)
meta['group'] = build_groups(meta, cross_class=CROSS_CLASS) if CROSS_CLASS else meta['session']
if DROP_PINK:
    meta = meta[meta['cls'] != 'Pink_disease'].reset_index(drop=True)
CLASSES = sorted(meta['cls'].unique(), key=CLASSES_ALL.index)
meta['y'] = meta['cls'].map({c: i for i, c in enumerate(CLASSES)})
missing = [(c, f) for c, f in zip(meta['cls'], meta['file']) if (c, f) not in cache['images']]
assert not missing, f'{len(missing)} files in sessions.csv were not found among the images, e.g. {missing[:3]}'
print(f'{len(meta)} images, {meta["group"].nunique()} groups, classes {CLASSES}')
print(meta.groupby('cls')['group'].nunique().rename('groups').to_string())

FOLDS_CSV = OUT_DIR / 'folds.csv'
if FOLDS_CSV.exists():
    folds = pd.read_csv(FOLDS_CSV)
    print(f'Reusing existing split: {FOLDS_CSV.name}')
else:
    rows = []
    for rep in range(N_REPEATS):
        sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=rep)
        for k, (tr, te) in enumerate(sgkf.split(meta, meta['y'], meta['group'])):
            # hold out a validation set from the training fold, by group
            # group sizes are very uneven (the largest has over a hundred images), so among the 10 candidates pick the one closest to 10% that has at least 3 classes
            inner = StratifiedGroupKFold(n_splits=round(1 / INNER_VAL_FRAC), shuffle=True,
                                         random_state=1000 + rep * 10 + k)
            sub = meta.iloc[tr]
            target = INNER_VAL_FRAC * len(sub)
            cands = [(abs(len(v) - target) + (1e6 if sub.iloc[v]['y'].nunique() < 3 else 0), t, v)
                     for t, v in inner.split(sub, sub['y'], sub['group'])]
            _, itr, iva = min(cands, key=lambda c: c[0])
            role = np.array(['test'] * len(meta), dtype=object)
            role[tr[itr]] = 'train'
            role[tr[iva]] = 'val'
            for idx in range(len(meta)):
                rows.append((rep, k, idx, role[idx]))
    folds = pd.DataFrame(rows, columns=['repeat', 'fold', 'idx', 'role'])
    folds.to_csv(FOLDS_CSV, index=False)
    print(f'Generated and saved split: {FOLDS_CSV.name}')

# Check: no group appears in both training and test
for (rep, k), f in folds.groupby(['repeat', 'fold']):
    g = {r: set(meta.loc[f[f.role == r].idx, 'group']) for r in ['train', 'val', 'test']}
    assert not (g['train'] & g['test']) and not (g['val'] & g['test']) and not (g['train'] & g['val']), (rep, k)
cnt = folds.groupby(['repeat', 'fold', 'role']).size().unstack()
print(f'Images per fold (mean): train {cnt.train.mean():.0f} / val {cnt.val.mean():.0f} / test {cnt.test.mean():.0f}, '
      'no group crosses train/val/test in any fold ✓')

## 5. Model (identical to train.py)

In [ ]:
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from torchvision.models import EfficientNet_B0_Weights

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


class LesionFocusAttention(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.attention_conv = nn.Conv2d(in_channels, 1, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return x * self.sigmoid(self.attention_conv(x))


class SEModule(nn.Module):
    def __init__(self, in_ch, reduction=16):
        super().__init__()
        mid = max(in_ch // reduction, 1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Linear(in_ch, mid, bias=False), nn.ReLU(inplace=True),
                                nn.Linear(mid, in_ch, bias=False), nn.Sigmoid())

    def forward(self, x):
        b, c, _, _ = x.size()
        return x * self.fc(self.pool(x).view(b, c)).view(b, c, 1, 1)


class CBAMModule(nn.Module):
    def __init__(self, in_ch, reduction=16, ks=7):
        super().__init__()
        mid = max(in_ch // reduction, 1)
        self.avg, self.mx = nn.AdaptiveAvgPool2d(1), nn.AdaptiveMaxPool2d(1)
        self.cfc = nn.Sequential(nn.Linear(in_ch, mid, bias=False), nn.ReLU(inplace=True),
                                 nn.Linear(mid, in_ch, bias=False))
        self.sconv = nn.Conv2d(2, 1, kernel_size=ks, padding=ks // 2, bias=False)
        self.sig = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.size()
        ch = self.sig(self.cfc(self.avg(x).view(b, c)) + self.cfc(self.mx(x).view(b, c))).view(b, c, 1, 1)
        x = x * ch
        sp = self.sig(self.sconv(torch.cat([x.mean(1, keepdim=True), x.max(1, keepdim=True)[0]], 1)))
        return x * sp


def build_model(attention, n_cls):
    m = models.efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT if PRETRAINED else None)
    for p in m.parameters():
        p.requires_grad = False
    last = m.features[-1][0].out_channels
    att = {'lfa': LesionFocusAttention, 'se': SEModule, 'cbam': CBAMModule}.get(attention)
    if att:
        m.features = nn.Sequential(m.features, att(last))
    m.classifier = nn.Sequential(nn.Dropout(p=0.3, inplace=True), nn.Linear(m.classifier[1].in_features, n_cls))
    for p in m.classifier.parameters():
        p.requires_grad = True
    if att:
        for p in m.features[-1].parameters():
            p.requires_grad = True
    return m


for a in ['none', 'lfa', 'se', 'cbam']:
    extra = sum(p.numel() for p in build_model(a, 4).parameters()) - sum(p.numel() for p in build_model('none', 4).parameters())
    print(f'{a:<5} extra parameters {extra:,}')

## 6. Training function (seeds actually take effect, checkpoint chosen by macro F1)

In [ ]:
import copy
from sklearn.metrics import f1_score, accuracy_score

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomAffine(degrees=20, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.GaussianBlur(kernel_size=(3, 7), sigma=(0.1, 1.5)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.10)),
])
test_tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
                              transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

PIL_CACHE = {k: Image.open(io.BytesIO(v)).convert('RGB') for k, v in cache['images'].items()}


class DS(Dataset):
    def __init__(self, idx, tf):
        self.idx, self.tf = list(idx), tf

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        r = meta.iloc[self.idx[i]]
        return self.tf(PIL_CACHE[(r['cls'], r['file'])]), int(r['y'])


def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def _worker_init(wid):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    yt, yp = [], []
    for X, y in loader:
        with torch.autocast(device_type='cuda', enabled=(DEVICE == 'cuda')):
            out = model(X.to(DEVICE))
        yp += out.argmax(1).cpu().tolist()
        yt += y.tolist()
    return np.array(yt), np.array(yp)


def macro_f1_present(yt, yp):
    """Macro F1 over only the classes present in the validation set (a small validation set may lack a class)."""
    return f1_score(yt, yp, labels=sorted(set(yt)), average='macro', zero_division=0)


def run_one(attention, tr_idx, va_idx, te_idx, seed):
    seed_everything(seed)
    n_cls = len(CLASSES)
    ytr = meta.iloc[tr_idx]['y'].values
    counts = np.bincount(ytr, minlength=n_cls)
    cw = torch.tensor(len(ytr) / (n_cls * np.maximum(counts, 1)), dtype=torch.float, device=DEVICE)
    sw = (1.0 / np.maximum(counts, 1))[ytr]
    g = torch.Generator().manual_seed(seed)
    sampler = WeightedRandomSampler(torch.as_tensor(sw, dtype=torch.double), len(sw), replacement=True, generator=g)
    kw = dict(num_workers=2, worker_init_fn=_worker_init, pin_memory=(DEVICE == 'cuda'))
    tl = DataLoader(DS(tr_idx, train_tf), batch_size=BATCH, sampler=sampler, generator=g, **kw)
    vl = DataLoader(DS(va_idx, test_tf), batch_size=64, shuffle=False, **kw)
    te = DataLoader(DS(te_idx, test_tf), batch_size=64, shuffle=False, **kw)

    model = build_model(attention, n_cls).to(DEVICE)
    crit = nn.CrossEntropyLoss(weight=cw)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))
    best = (-1.0, None)
    e1, e2 = (1, 1) if QUICK_TEST else (EPOCHS_S1, EPOCHS_S2)

    def epochs(n, lr):
        nonlocal best
        opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
        for _ in range(n):
            model.train()
            for X, y in tl:
                X, y = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                with torch.autocast(device_type='cuda', enabled=(DEVICE == 'cuda')):
                    loss = crit(model(X), y)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            f1v = macro_f1_present(*predict(model, vl))
            if f1v > best[0]:
                best = (f1v, copy.deepcopy(model.state_dict()))

    epochs(e1, 1e-3)                       # Stage 1: frozen backbone, train only attention and classifier head
    model.load_state_dict(best[1])
    for p in model.parameters():
        p.requires_grad = True
    epochs(e2, 1e-5)                       # Stage 2: unfreeze everything
    model.load_state_dict(best[1])
    yt, yp = predict(model, te)
    return {
        'val_macro_f1': round(best[0] * 100, 2),
        'test_macro_f1': round(f1_score(yt, yp, labels=list(range(n_cls)), average='macro', zero_division=0) * 100, 2),
        'test_acc': round(accuracy_score(yt, yp) * 100, 2),
    }, yt, yp

## 7. Main loop (interruptible and resumable)

In [ ]:
RUN_DIR = OUT_DIR / ('quick_test' if QUICK_TEST else 'main')   # trial-run results go in a separate folder, so they don't mix with the real results
RUN_DIR.mkdir(parents=True, exist_ok=True)
RESULTS = RUN_DIR / 'results.csv'
PREDS = RUN_DIR / 'preds.csv'
done = set()
if RESULTS.exists():
    r0 = pd.read_csv(RESULTS)
    done = set(zip(r0.config, r0.repeat, r0.fold, r0.seed))
    print(f'{len(done)} runs already done, continuing...')

fold_keys = sorted(folds.groupby(['repeat', 'fold']).groups)
if QUICK_TEST:
    fold_keys = fold_keys[:2]
jobs = [(cfg, rep, k, s) for phase in PHASES for (rep, k) in fold_keys for s in SEEDS for cfg in phase]
todo = [j for j in jobs if j not in done]
print(f'{len(jobs)} runs in total, {len(todo)} remaining')

t_start, n_run = time.time(), 0
for cfg, rep, k, s in todo:
    f = folds[(folds.repeat == rep) & (folds.fold == k)]
    tr, va, te = (f[f.role == r].idx.values for r in ['train', 'val', 'test'])
    t0 = time.time()
    res, yt, yp = run_one(cfg, tr, va, te, s)
    row = {'config': cfg, 'repeat': rep, 'fold': k, 'seed': s,
           'n_train': len(tr), 'n_val': len(va), 'n_test': len(te), **res,
           'minutes': round((time.time() - t0) / 60, 2)}
    pd.DataFrame([row]).to_csv(RESULTS, mode='a', header=not RESULTS.exists(), index=False)
    pr = pd.DataFrame({'config': cfg, 'repeat': rep, 'fold': k, 'seed': s,
                       'file': meta.iloc[te]['file'].values, 'cls': meta.iloc[te]['cls'].values,
                       'y_true': yt, 'y_pred': yp})
    pr.to_csv(PREDS, mode='a', header=not PREDS.exists(), index=False)
    n_run += 1
    per = (time.time() - t_start) / n_run
    left = (len(todo) - n_run) * per / 3600
    print(f'[{len(done) + n_run}/{len(jobs)}] {cfg:<5} rep{rep} fold{k} seed{s}  '
          f'test F1 {res["test_macro_f1"]:5.1f}  ({row["minutes"]:.1f} min)  about {left:.1f} h left', flush=True)
print('All done ✓')

## 8. Analysis (run this cell in the morning and send the output to Claude)
Decision rule (fixed before running, not to be changed): mean difference LFA - no attention >= 2 pp, **and** the lower bound of the corrected 95% CI > 0.

In [ ]:
import numpy as np, pandas as pd
from scipy import stats

r = pd.read_csv(RESULTS)
print(f'{RESULTS} {len(r)} training runs done\n')
print('Summary per config (one number per training run):')
print(r.groupby('config')['test_macro_f1'].agg(['count', 'mean', 'std']).round(2).to_string())

per_fold = r.groupby(['config', 'repeat', 'fold'])['test_macro_f1'].mean().unstack('config')
n_tr = r['n_train'].mean() + r['n_val'].mean()
n_te = r['n_test'].mean()


def corrected_ttest(d, n_train, n_test):
    """Nadeau & Bengio (2003) corrected resampled t-test."""
    J = len(d)
    m, v = d.mean(), d.var(ddof=1)
    se = np.sqrt((1 / J + n_test / n_train) * v)
    t = m / se
    p = 2 * stats.t.sf(abs(t), J - 1)
    h = stats.t.ppf(0.975, J - 1) * se
    return m, (m - h, m + h), p


print('\nPaired comparison (seeds averaged within each fold first, then compared):')
print(f'{"comparison":<16}{"folds":>5}{"mean diff":>10}{"corrected 95% CI":>20}{"corr. p":>9}{"uncorr. p":>10}{"folds won":>10}')
for other in ['lfa', 'se', 'cbam']:
    if other not in per_fold or 'none' not in per_fold:
        continue
    d = (per_fold[other] - per_fold['none']).dropna()
    if len(d) < 3:
        continue
    m, ci, p = corrected_ttest(d.values, n_tr, n_te)
    p_naive = stats.ttest_1samp(d.values, 0).pvalue
    print(f'{other + " − none":<16}{len(d):>5}{m:>8.2f}   [{ci[0]:6.2f}, {ci[1]:6.2f}]{p:>9.3f}{p_naive:>10.3f}'
          f'{f"{(d > 0).sum()}/{len(d)}":>9}')
    if other == 'lfa':
        verdict = 'A (LFA clearly improves)' if (m >= 2 and ci[0] > 0) else 'B (no reliable improvement from LFA)'
        print(f'{"":16}→ per the pre-set decision rule: branch {verdict}')

# What a single experiment would report: pick a random (LFA, none) pair of runs; probability that LFA leads by >= 3 pp
a = r[r.config == 'lfa'].set_index(['repeat', 'fold', 'seed'])['test_macro_f1']
b = r[r.config == 'none'].set_index(['repeat', 'fold', 'seed'])['test_macro_f1']
pair = (a - b).dropna()
if len(pair):
    print(f'\nAmong single paired runs, share where LFA leads by >= 3 pp: {(pair >= 3).mean() * 100:.0f}%; '
          f'share where it trails by >= 3 pp: {(pair <= -3).mean() * 100:.0f}% ({len(pair)} pairs in total)')
same = r.groupby(['config', 'repeat', 'fold'])['test_macro_f1'].agg(lambda x: x.max() - x.min())
print(f'Same config, same fold, only the seed changed: results differ by {same.mean():.2f} pp on average')